# LFM2.5-1.2B Fine-Tuning — Mininio Carb Assistant

Thin Colab wrapper. All logic lives in `finetuning/lfm/train_lfm.py`.

**Prerequisites:**
1. Upload `data/output/lfm/` (train.jsonl + eval.jsonl) as `mininio-data.zip` to the root of your Google Drive
2. The zip contents should be `lfm/train.jsonl` and `lfm/eval.jsonl` (no extra wrapper directory)
3. Create with: `cd data/output && zip -r mininio-data.zip lfm/ gemma/`

**Runtime:** Colab Free T4 (16GB) — ~40-70 min for 3 epochs
---

In [ ]:
# Install dependencies (Colab-specific pins from Unsloth reference)
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth==2026.7.3
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v == "2.9" else "0.0.32.post2" if v == "2.8" else "0.0.29.post3")
    !pip install -q --no-deps bitsandbytes==0.49.2 accelerate==1.14.0 {xformers} peft==0.19.1 trl==0.24.0 triton cut_cross_entropy==25.1.1 unsloth_zoo==2026.7.3 transformers==5.5.0
    !pip install -q sentencepiece==0.2.2 protobuf "datasets==4.3.0" "huggingface_hub==1.23.0" hf_transfer
    !pip install -q --no-deps unsloth==2026.7.3
!pip install -q loguru==0.7.3

# HF token — set in Colab secrets (key icon in left sidebar) or paste below
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
# WandB — set in Colab secrets or paste below; leave empty to skip logging
os.environ["WANDB_API_KEY"] = os.environ.get("WANDB_API_KEY", "")
os.environ["WANDB_PROJECT"] = os.environ.get("WANDB_PROJECT", "mininio-ai-finetuning")
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 12.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.3 requires msgspec, which is not installed.
unsloth-zoo 2026.7.3 requires tyro; sys_platform != "darwin" or platform_machine != "a

In [2]:
# Clear stale Unsloth compiled cache (must match pinned transformers version)
import shutil, os
cache = "/content/unsloth_compiled_cache"
if os.path.exists(cache):
    shutil.rmtree(cache)
    print("Cleared stale Unsloth compiled cache.")
print("Ready.")

Ready.


In [3]:
# Clone repo & mount Drive, copy data
%cd /content

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/mininio-ai-finetuning'
REPO_URL = 'https://github.com/CoGian/mininio-ai-finetuning.git'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('Pulling latest...')
    !git -C "{REPO_DIR}" pull
else:
    print('Cloning repo...')
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

!cp /content/drive/MyDrive/mininio-data.zip /content/ 2>/dev/null
!mkdir -p /content/data/output
!unzip -o /content/mininio-data.zip -d /content/data/output/ 2>/dev/null
!ls /content/data/output/lfm/ 2>/dev/null || echo "Data not found. Make sure mininio-data.zip is in Drive root."

Mounted at /content/drive
Cloning repo...
Cloning into '/content/mininio-ai-finetuning'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 78 (delta 2), reused 44 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 302.77 KiB | 10.44 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Archive:  /content/mininio-data.zip
  inflating: /content/data/output/lfm/eval.jsonl  
  inflating: /content/data/output/lfm/train.jsonl  
  inflating: /content/data/output/gemma/eval.jsonl  
  inflating: /content/data/output/gemma/train.jsonl  
eval.jsonl  train.jsonl


In [ ]:
# Run training (3 epochs, max_seq_length 4096, QLoRA r16, effective batch 32)
!PYTHONPATH=/content/mininio-ai-finetuning python -m finetuning.lfm.train_lfm \
    --data-dir /content/data/output \
    --output-dir /content/drive/MyDrive/mininio-checkpoints \
    --epochs 3 \
    --max-seq-length 4096 \
    --batch-size 8 \
    --grad-accum 4 \

print("\nCheckpoints saved to Drive: /content/drive/MyDrive/mininio-checkpoints/lfm/")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026-08-05 16:02:49.340 | INFO     | __main__:main:46 - Report to: wandb
2026-08-05 16:02:49.341 | INFO     | __main__:main:66 - Environment: colab
2026-08-05 16:02:49.341 | INFO     | __main__:main:67 - Data dir: /content/data/output/lfm
2026-08-05 16:02:49.342 | INFO     | __main__:main:68 - Output dir: /content/drive/MyDrive/mininio-checkpoints/lfm
2026-08-05 16:02:49.342 | INFO     | __main__:main:69 - Training config: bs=8 ga=4 epochs=3 lr=0.0002 max_seq=4096 seed=3407
==((====))==  Unsloth 2026.7.3: Fast Lfm2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast download

### Smoke Test (optional)
Run a quick inference test to verify the model works.

In [ ]:
# Quick smoke test
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/mininio-checkpoints/lfm/merged_16bit",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "I ate 100g of potatoes. How many carbs?"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_k=50,
    top_p=0.1,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)